In [2]:
import numpy as np
import plotly.express as px
import pandas as pd
from sklearn.decomposition import PCA
from modules.benchmark import *

In [3]:
visco = quick_load(feature='expert_chen', columns=['smiles_surfactant1', 'smiles_surfactant2', 'smiles_additive1', 'smiles_additive2'])

In [5]:
X, y, groups = xy_split(visco)

In [7]:
X = X.drop(columns=['type_surfactant1', 'type_surfactant2'])
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8185 entries, 0 to 8184
Data columns (total 89 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   c_surfactant1                       8185 non-null   float64
 1   c_surfactant2                       8185 non-null   float64
 2   c_additive1                         8185 non-null   float64
 3   c_additive2                         8185 non-null   float64
 4   temperature                         8185 non-null   int64  
 5   surfactant1_chen_MolLogP            8185 non-null   float64
 6   surfactant1_chen_Chi1v              8185 non-null   float64
 7   surfactant1_chen_VSA_EState7        8185 non-null   float64
 8   surfactant1_chen_VSA_EState8        8185 non-null   float64
 9   surfactant1_chen_qed                8185 non-null   float64
 10  surfactant1_chen_FpDensityMorgan1   8185 non-null   float64
 11  surfactant1_chen_SMR_VSA5           8185 no

In [11]:
X = X.fillna(-1)

In [12]:
embeddings = X.to_numpy()

In [13]:
embeddings

array([[395. ,  55. , 100. , ...,  -1. ,  -1. ,  -1. ],
       [395. ,  55. , 150. , ...,  -1. ,  -1. ,  -1. ],
       [395. ,  55. , 200. , ...,  -1. ,  -1. ,  -1. ],
       ...,
       [ 50. ,  22.5, 500. , ...,  -1. ,  -1. ,  -1. ],
       [ 50. ,  25. , 500. , ...,  -1. ,  -1. ,  -1. ],
       [ 50. ,  27.5, 500. , ...,  -1. ,  -1. ,  -1. ]], shape=(8185, 89))

In [14]:
pca = PCA(n_components=2)
pca_embeddings = pca.fit_transform(embeddings)

pca_df = pd.DataFrame({
    "PC1": pca_embeddings[:, 0],
    "PC2": pca_embeddings[:, 1],
    "label": y,
})

fig = px.scatter(
    pca_df,
    x="PC1",
    y="PC2",
    color="label",
    hover_name="label",
    title="PCA projection of image embeddings",
    labels={
        "PC1": f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)",
        "PC2": f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)",
    },
)
fig.show()

In [15]:
embeddings.shape[1]

89

In [13]:
def plot_explained_variance_by_dimension(embeddings: np.ndarray):
    pca_full = PCA(n_components=min(embeddings.shape), random_state=42)
    pca_full.fit(embeddings)

    variance_df = pd.DataFrame({
        "dimension": np.arange(1, len(pca_full.explained_variance_ratio_) + 1),
        "explained_variance": pca_full.explained_variance_ratio_,
        "cumulative_explained_variance": np.cumsum(pca_full.explained_variance_ratio_),
    })

    fig = px.bar(
        variance_df,
        x="dimension",
        y="explained_variance",
        title="PCA Explained Variance by Dimension",
        labels={
            "dimension": "PCA dimension",
            "explained_variance": "Explained variance ratio",
        },
    )

    fig.add_scatter(
        x=variance_df["dimension"],
        y=variance_df["cumulative_explained_variance"],
        mode="lines+markers",
        name="Cumulative explained variance",
        yaxis="y2",
    )

    fig.update_layout(
        xaxis_title="PCA dimension",
        yaxis=dict(
            title="Explained variance ratio",
            tickformat=".0%",
        ),
        yaxis2=dict(
            title="Cumulative explained variance",
            tickformat=".0%",
            overlaying="y",
            side="right",
            range=[0, 1.05],
        ),
        legend_title_text="",
        hovermode="x unified",
    )

    fig.show()
plot_explained_variance_by_dimension(embeddings)

## Without concentration and temperature

In [18]:
embeddings = embeddings[:,6:]

In [19]:
pca = PCA(n_components=2)
pca_embeddings = pca.fit_transform(embeddings)

pca_df = pd.DataFrame({
    "PC1": pca_embeddings[:, 0],
    "PC2": pca_embeddings[:, 1],
    "label": y,
})

fig = px.scatter(
    pca_df,
    x="PC1",
    y="PC2",
    color="label",
    hover_name="label",
    title="PCA projection of image embeddings",
    labels={
        "PC1": f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)",
        "PC2": f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)",
    },
)
fig.show()

In [20]:
plot_explained_variance_by_dimension(embeddings)

In [21]:
print(pca.components_.shape)

(2, 83)


In [22]:
print(pca.components_[0])

[ 1.56207089e-02  3.10915325e-02 -1.54409223e-03 -5.22475234e-05
 -4.49645449e-05  5.36000064e-02 -7.40629190e-03  1.06644202e-04
  7.61301404e-01  2.86391261e-03  1.32596147e-03  3.96452899e-04
  1.67895431e-01 -7.07321448e-04 -9.17907102e-03 -3.28612973e-05
  3.71446488e-02  2.34434102e-04  6.19163532e-01 -2.14372084e-03
  3.67759732e-04  7.48195625e-04  9.95695116e-05  6.82512613e-04
  7.71917436e-05  9.66520420e-05  5.04136046e-03  3.80857819e-03
  3.34346093e-05  3.62063279e-02  1.86776796e-03  7.52346331e-04
  2.48613122e-04  3.97818599e-03 -1.03035301e-05  3.21593300e-03
  7.36244986e-05  6.88804737e-04  1.00696649e-03  2.79921262e-02
  7.29962610e-04  1.49396618e-04 -4.11619377e-04 -2.47949127e-04
 -3.27174827e-04 -1.39244424e-04 -2.20324282e-04 -1.26342538e-04
 -1.35551390e-03 -6.56421759e-05 -2.59110820e-02 -9.75571429e-04
 -1.29802998e-03 -8.72232776e-05 -1.54542534e-03 -3.29736106e-05
 -1.61559165e-03 -2.14387200e-04 -1.44940607e-03 -7.00776699e-04
 -1.98309840e-02 -1.00697

In [23]:
numb = []
for i in range(len(pca.components_[0])):
    numb.append((i, pca.components_[0][i]))

numb.sort(key=lambda x: x[1])

In [24]:
numb[:10]

[(50, np.float64(-0.025911082040236642)),
 (60, np.float64(-0.019830984006558534)),
 (14, np.float64(-0.0091790710209224)),
 (6, np.float64(-0.007406291898845262)),
 (19, np.float64(-0.002143720841736473)),
 (56, np.float64(-0.0016155916497893068)),
 (54, np.float64(-0.001545425342531684)),
 (2, np.float64(-0.0015440922301516793)),
 (81, np.float64(-0.00153714108969986)),
 (58, np.float64(-0.0014494060681885651))]

# PCA for single compounds - makes more sense

In [3]:
visco = load_cleaned()
visco.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8185 entries, 0 to 8184
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_surfactant1      8185 non-null   object 
 1   type_surfactant1    8185 non-null   object 
 2   smiles_surfactant1  8185 non-null   object 
 3   c_surfactant1       8185 non-null   float64
 4   id_surfactant2      2727 non-null   object 
 5   type_surfactant2    2727 non-null   object 
 6   smiles_surfactant2  2632 non-null   object 
 7   c_surfactant2       8185 non-null   float64
 8   id_additive1        5519 non-null   object 
 9   smiles_additive1    5519 non-null   object 
 10  c_additive1         8185 non-null   float64
 11  id_additive2        1279 non-null   object 
 12  smiles_additive2    1123 non-null   object 
 13  c_additive2         8185 non-null   float64
 14  temperature         8185 non-null   int64  
 15  viscoelasticity     8185 non-null   float64
 16  g_0   

In [4]:
surf1 = visco['smiles_surfactant1']

In [6]:
surf1 = surf1.to_frame(name='smiles')
surf1

,smiles
0,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+]
1,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+]
2,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+]
3,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+]
4,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+]
...,...
8180,CCCCCCCCCCCCCC[N+](C)(C)CCCS(=O)(=O)[O-]
8181,CCCCCCCCCCCCCC[N+](C)(C)CCCS(=O)(=O)[O-]
8182,CCCCCCCCCCCCCC[N+](C)(C)CCCS(=O)(=O)[O-]
8183,CCCCCCCCCCCCCC[N+](C)(C)CCCS(=O)(=O)[O-]


In [7]:
surf1 = add_expert_chen(surf1, columns=['smiles'])
surf1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8185 entries, 0 to 8184
Data columns (total 22 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   smiles                         8185 non-null   object 
 1   smiles_chen_MolLogP            8185 non-null   float64
 2   smiles_chen_Chi1v              8185 non-null   float64
 3   smiles_chen_VSA_EState7        8185 non-null   float64
 4   smiles_chen_VSA_EState8        8185 non-null   float64
 5   smiles_chen_qed                8185 non-null   float64
 6   smiles_chen_FpDensityMorgan1   8185 non-null   float64
 7   smiles_chen_SMR_VSA5           8185 non-null   float64
 8   smiles_chen_PEOE_VSA6          8185 non-null   float64
 9   smiles_chen_MinPartialCharge   8185 non-null   float64
 10  smiles_chen_BertzCT            8185 non-null   float64
 11  smiles_chen_VSA_EState1        8185 non-null   float64
 12  smiles_chen_MaxAbsEStateIndex  8185 non-null   f

In [8]:
embeddings = surf1.drop('smiles', axis=1).to_numpy()

In [10]:
y = np.zeros(embeddings.shape[0])

In [11]:
pca = PCA(n_components=2)
pca_embeddings = pca.fit_transform(embeddings)

pca_df = pd.DataFrame({
    "PC1": pca_embeddings[:, 0],
    "PC2": pca_embeddings[:, 1],
    "label": y,
})

fig = px.scatter(
    pca_df,
    x="PC1",
    y="PC2",
    color="label",
    hover_name="label",
    title="PCA projection of image embeddings",
    labels={
        "PC1": f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)",
        "PC2": f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)",
    },
)
fig.show()

In [14]:
plot_explained_variance_by_dimension(embeddings)

In [15]:
numb = []
for i in range(len(pca.components_[0])):
    numb.append((i, pca.components_[0][i]))

numb.sort(key=lambda x: x[1])

In [16]:
numb[:10]

[(15, np.float64(-0.008810866484844212)),
 (7, np.float64(-0.00721250874447806)),
 (20, np.float64(-0.002124938461004178)),
 (3, np.float64(-0.001372040557551648)),
 (14, np.float64(-0.0007098483428953741)),
 (4, np.float64(-5.100304292431629e-05)),
 (5, np.float64(-4.629043746864758e-05)),
 (16, np.float64(-3.8000322946578796e-05)),
 (8, np.float64(0.00010426705045203164)),
 (18, np.float64(0.0002436381129243268))]

In [29]:
add1 = visco['smiles_additive1'].dropna()

In [30]:
add1 = add1.to_frame(name='smiles')
add1

,smiles
0,[Cl-].[Na+]
1,[Cl-].[Na+]
2,[Cl-].[Na+]
3,[Cl-].[Na+]
4,[Cl-].[Na+]
...,...
8180,[Cl-].[Na+]
8181,[Cl-].[Na+]
8182,[Cl-].[Na+]
8183,[Cl-].[Na+]


In [31]:
add1 = add_expert_chen(add1, columns=['smiles'])

In [32]:
embeddings = add1.drop('smiles', axis=1).to_numpy()
y = np.zeros(embeddings.shape[0])

In [33]:
pca = PCA(n_components=2)
pca_embeddings = pca.fit_transform(embeddings)

pca_df = pd.DataFrame({
    "PC1": pca_embeddings[:, 0],
    "PC2": pca_embeddings[:, 1],
    "label": y,
})

fig = px.scatter(
    pca_df,
    x="PC1",
    y="PC2",
    color="label",
    hover_name="label",
    title="PCA projection of image embeddings",
    labels={
        "PC1": f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)",
        "PC2": f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)",
    },
)
fig.show()

In [34]:
plot_explained_variance_by_dimension(embeddings)

In [35]:
numb = []
for i in range(len(pca.components_[0])):
    numb.append((i, pca.components_[0][i]))

numb.sort(key=lambda x: x[1])

In [36]:
numb[:10]

[(14, np.float64(-0.005759365826739626)),
 (3, np.float64(-0.001894865635446117)),
 (6, np.float64(-0.0015368831914917493)),
 (20, np.float64(-0.00045607148840076574)),
 (16, np.float64(-0.00028546854499057734)),
 (12, np.float64(0.0006117766754116775)),
 (8, np.float64(0.0006487388927745766)),
 (5, np.float64(0.0007014800846572162)),
 (4, np.float64(0.0007901181901138108)),
 (2, np.float64(0.0034730955974830225))]

# Avalon

In [37]:
surf1 = visco['smiles_surfactant1']
surf1 = surf1.to_frame(name='smiles')
surf1 = add_avalon(surf1, columns=['smiles'])
embeddings = surf1.drop('smiles', axis=1).to_numpy()
y = np.zeros(embeddings.shape[0])

In [38]:
pca = PCA(n_components=2)
pca_embeddings = pca.fit_transform(embeddings)

pca_df = pd.DataFrame({
    "PC1": pca_embeddings[:, 0],
    "PC2": pca_embeddings[:, 1],
    "label": y,
})

fig = px.scatter(
    pca_df,
    x="PC1",
    y="PC2",
    color="label",
    hover_name="label",
    title="PCA projection of image embeddings",
    labels={
        "PC1": f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)",
        "PC2": f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)",
    },
)
fig.show()

In [39]:
plot_explained_variance_by_dimension(embeddings)

In [40]:
numb = []
for i in range(len(pca.components_[0])):
    numb.append((i, pca.components_[0][i]))

numb.sort(key=lambda x: x[1])
numb[:10]

[(16, np.float64(-0.15243485715254573)),
 (98, np.float64(-0.1455943662462976)),
 (151, np.float64(-0.1455943662462976)),
 (118, np.float64(-0.1453256034214082)),
 (62, np.float64(-0.13483740605525601)),
 (108, np.float64(-0.13102046503038003)),
 (50, np.float64(-0.1277751693645935)),
 (69, np.float64(-0.12244798813848207)),
 (171, np.float64(-0.12176757102273958)),
 (132, np.float64(-0.11991815071961247))]

# MACCS

In [41]:
surf1 = visco['smiles_surfactant1']
surf1 = surf1.to_frame(name='smiles')
surf1 = add_maccs(surf1, columns=['smiles'])
embeddings = surf1.drop('smiles', axis=1).to_numpy()
plot_explained_variance_by_dimension(embeddings)

# BCUT2d

In [42]:
surf1 = visco['smiles_surfactant1']
surf1 = surf1.to_frame(name='smiles')
surf1 = add_bcut2d(surf1, columns=['smiles'])
embeddings = surf1.drop('smiles', axis=1).to_numpy()
plot_explained_variance_by_dimension(embeddings)

# Physiochemical

In [43]:
surf1 = visco['smiles_surfactant1']
surf1 = surf1.to_frame(name='smiles')
surf1 = add_physiochemical(surf1, columns=['smiles'])
embeddings = surf1.drop('smiles', axis=1).to_numpy()
plot_explained_variance_by_dimension(embeddings)